In [1]:
import numpy as np
import pandas as pd
import re

In [2]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, explained_variance_score, max_error, median_absolute_error

In [3]:
import mlflow
from mlflow.models.signature import infer_signature

mlflow.set_experiment("MLP_Regressor_HPO")
mlflow.set_tracking_uri("http://localhost:5000")

In [4]:
import json
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from optuna.samplers import TPESampler
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, HalvingRandomSearchCV

/home/elianaostro/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
param_list = {}

In [6]:
# Define a function to calculate all metrics and format a report
def calculate_metrics(y, y_pred, y_val = None, y_pred_val = None, verbose=True):
    # Calculate train metrics
    mse = mean_squared_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    mae = mean_absolute_error(y, y_pred)
    evs = explained_variance_score(y, y_pred)
    max_err = max_error(y, y_pred)
    medae = median_absolute_error(y, y_pred)
    metrics = {
        "mse": mse,
        "r2": r2,
        "mae": mae,
        "evs": evs,
        "max_err": max_err,
        "medae": medae
    }

    # If validation data is provided, calculate validation metrics
    if y_val is not None and y_pred_val is not None:
        # Calculate validation metrics
        mse_val = mean_squared_error(y_val, y_pred_val)
        r2_val = r2_score(y_val, y_pred_val)
        mae_val = mean_absolute_error(y_val, y_pred_val)
        evs_val = explained_variance_score(y_val, y_pred_val)
        max_err_val = max_error(y_val, y_pred_val)
        medae_val = median_absolute_error(y_val, y_pred_val)
        metrics.update({
            "mse_val": mse_val,
            "r2_val": r2_val,
            "mae_val": mae_val,
            "evs_val": evs_val,
            "max_err_val": max_err_val,
            "medae_val": medae_val
        })

        # Create a table for formatting metrics
        if verbose:
            print("Metric       |    Train     |  Validation")
            print("-------------|--------------|---------------")
            print(f"MSE          |   {mse:.2e}   |   {mse_val:.2e}")
            print(f"R²           |   {r2:.4f}     |   {r2_val:.4f}")
            print(f"MAE          |   {mae:.2e}   |   {mae_val:.2e}")
            print(f"EVS          |   {evs:.4f}     |   {evs_val:.4f}")
            print(f"Max Error    |   {max_err:.2e}   |   {max_err_val:.2e}")
            print(f"Median AE    |   {medae:.2e}   |   {medae_val:.2e}")
            print()
            
    elif verbose:
        print(f"MSE          |   {mse:.2e}   ")
        print(f"R²           |   {r2:.4f}     ")
        print(f"MAE          |   {mae:.2e}   ")
        print(f"EVS          |   {evs:.4f}     ")
        print(f"Max Error    |   {max_err:.2e}   ")
        print(f"Median AE    |   {medae:.2e}   ")
        print()
    
    # Return metrics in a dictionary for reporting
    return metrics

## Preparacion de datos

In [7]:
df_train = pd.read_csv('../data/train/transformed_train.csv')
df_train.info()

from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='mean')  # o 'median', 'most_frequent', etc.
df_train_imputed = imputer.fit_transform(df_train)
df_train = pd.DataFrame(df_train_imputed, columns=df_train.columns)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11682 entries, 0 to 11681
Data columns (total 81 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Marca                              11682 non-null  float64
 1   Modelo                             11682 non-null  float64
 2   Versión                            11682 non-null  float64
 3   Motor                              11664 non-null  float64
 4   Turbo                              11682 non-null  int64  
 5   cv                                 6145 non-null   float64
 6   Puertas                            11682 non-null  float64
 7   Kilómetros                         11682 non-null  float64
 8   Precio                             11682 non-null  float64
 9   Año                                11682 non-null  float64
 10  desc_new_svd_1                     11682 non-null  float64
 11  desc_new_svd_2                     11682 non-null  flo

In [8]:
df_val = pd.read_csv('../data/train/transformed_val.csv')
df_val.info()

df_val_imputed = imputer.transform(df_val)
df_val = pd.DataFrame(df_val_imputed, columns=df_val.columns)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2921 entries, 0 to 2920
Data columns (total 81 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Marca                              2921 non-null   float64
 1   Modelo                             2921 non-null   float64
 2   Versión                            2921 non-null   float64
 3   Motor                              2919 non-null   float64
 4   Turbo                              2921 non-null   int64  
 5   cv                                 1496 non-null   float64
 6   Puertas                            2921 non-null   float64
 7   Kilómetros                         2921 non-null   float64
 8   Precio                             2921 non-null   float64
 9   Año                                2921 non-null   float64
 10  desc_new_svd_1                     2921 non-null   float64
 11  desc_new_svd_2                     2921 non-null   float

In [9]:
print("Train data after imputation:", df_train.isna().sum().sum())
print("Validation data after imputation:", df_val.isna().sum().sum())

Train data after imputation: 0
Validation data after imputation: 0


In [10]:
# Variable target
y_train = df_train["Precio"]
X_train = df_train.drop(columns=["Precio"])

X_val = df_val.drop(columns=["Precio"])
y_val = df_val["Precio"]

In [11]:
print("Train data after imputation:", X_train.isna().sum().sum())
print("Validation data after imputation:", X_val.isna().sum().sum())

Train data after imputation: 0
Validation data after imputation: 0


## First Experiment

Ajuste de hiperparametros

In [ ]:
# Espacio de hiperparámetros refinado y más estable para MLPRegressor
param_dist = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam'],
    'alpha': [1e-3, 1e-2],
    'learning_rate': ['constant', 'adaptive'],
    'learning_rate_init': [0.001, 0.005],
    'max_iter': [500, 1000]
}

# Modelo base con early stopping
mlp = MLPRegressor(
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10
)

# RandomizedSearchCV
search = RandomizedSearchCV(
    estimator=mlp,
    param_distributions=param_dist,
    n_iter=30,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Entrenar
search.fit(X_train, np.log1p(y_train))  # log-transform del target
best_model = search.best_estimator_
best_params = search.best_params_

Fitting 3 folds for each of 30 candidates, totalling 90 fits


In [ ]:
# Predicciones
y_pred_train = np.expm1(best_model.predict(X_train))
y_pred_val = np.expm1(best_model.predict(X_val))

calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)

param_list["RandomizedSearchCV"] = best_params

print("Best parameters found:")
print(json.dumps(best_params, indent=4))


Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   1.01e+14   |   8.97e+13
R²           |   0.7998     |   0.8017
MAE          |   3.96e+06   |   4.21e+06
EVS          |   0.8005     |   0.8018
Max Error    |   5.98e+08   |   2.31e+08
Median AE    |   2.08e+06   |   2.20e+06

Best parameters found:
{
    "solver": "adam",
    "max_iter": 500,
    "learning_rate_init": 0.001,
    "learning_rate": "constant",
    "hidden_layer_sizes": [
        100
    ],
    "alpha": 0.01,
    "activation": "tanh"
}


In [ ]:
# Espacio de hiperparámetros refinado y más estable para MLPRegressor
param_dist = {
    'hidden_layer_sizes': [
        (100, 100),
        (150, 100),
        (200, 100),
        (100, 100, 50),
        (200, 150, 100),
        (300, 200, 100)
    ],
    'activation': ['relu', 'tanh', 'logistic'],
    'solver': ['adam'],
    'alpha': [1e-4, 1e-3, 1e-2, 0.05],
    'learning_rate': ['constant', 'adaptive'],
    'learning_rate_init': [0.0005, 0.001, 0.005, 0.01],
    'max_iter': [1000, 1500],
    'early_stopping': [True],
    'validation_fraction': [0.1],
    'n_iter_no_change': [10, 15],
    'batch_size': ['auto', 64, 128]
}

# Modelo base con early stopping
mlp = MLPRegressor(
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10
)

# RandomizedSearchCV
search = RandomizedSearchCV(
    estimator=mlp,
    param_distributions=param_dist,
    n_iter=30,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Entrenar
search.fit(X_train, np.log1p(y_train))  # log-transform del target
best_model = search.best_estimator_
best_params = search.best_params_

# Predicciones
y_pred_train = np.expm1(best_model.predict(X_train))
y_pred_val = np.expm1(best_model.predict(X_val))

calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)

param_list["RandomizedSearchCV"] = best_params

print("Best parameters found:")
print(json.dumps(best_params, indent=4))


Fitting 3 folds for each of 30 candidates, totalling 90 fits
Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   1.16e+14   |   8.28e+13
R²           |   0.7709     |   0.8169
MAE          |   4.26e+06   |   4.28e+06
EVS          |   0.7716     |   0.8171
Max Error    |   6.02e+08   |   2.48e+08
Median AE    |   2.26e+06   |   2.30e+06

Best parameters found:
{
    "validation_fraction": 0.1,
    "solver": "adam",
    "n_iter_no_change": 10,
    "max_iter": 1500,
    "learning_rate_init": 0.01,
    "learning_rate": "constant",
    "hidden_layer_sizes": [
        100,
        100
    ],
    "early_stopping": true,
    "batch_size": 64,
    "alpha": 0.001,
    "activation": "logistic"
}


In [ ]:
def objective(trial):
    params = {
        'hidden_layer_sizes': (100,100),
        'activation': trial.suggest_categorical('activation', ['relu', 'tanh']),
        'solver': 'adam',
        'alpha': trial.suggest_float('alpha', 1e-4, 1e-2, log=True),
        'learning_rate': trial.suggest_categorical('learning_rate', ['constant', 'adaptive']),
        'learning_rate_init': trial.suggest_float('learning_rate_init', 0.001, 0.01, log=True),
        'max_iter': trial.suggest_int('max_iter', 500, 1000),
        'early_stopping': True,
        'validation_fraction': 0.1,
        'n_iter_no_change': 10,
        'random_state': 42
    }

    model = MLPRegressor(**params)
    scores = cross_val_score(
        model,
        X_train,
        np.log1p(y_train),
        cv=3,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1
    )
    return -scores.mean()


optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(direction="minimize", sampler=TPESampler())
study.optimize(objective, n_trials=50)
best_params = study.best_params

# Ajustar mejor modelo con los mejores parámetros
best_params['early_stopping'] = True
best_params['validation_fraction'] = 0.1
best_params['n_iter_no_change'] = 10
best_params['solver'] = 'adam'
best_params['random_state'] = 42
best_params['hidden_layer_sizes'] = (100, 100)

best_model = MLPRegressor(**best_params)
best_model.fit(X_train, np.log1p(y_train))

# Predicciones y destransformación
y_pred_train = np.expm1(best_model.predict(X_train))
y_pred_val = np.expm1(best_model.predict(X_val))

# Reporte
calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)
param_list["Optuna_MLP"] = best_params

print("Best parameters found:")
print(json.dumps(best_params, indent=4))



In [ ]:
param_dist = {
    'hidden_layer_sizes': [(100,100), (100, 200), (200, 200), (500, 100)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam'],
    'alpha': [1e-3, 1e-2],
    'learning_rate': ['constant', 'adaptive'],
    'learning_rate_init': [0.001, 0.005],
    'max_iter': [500, 1000]
}

mlp = MLPRegressor(
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10
)

halving_search = HalvingRandomSearchCV(
    estimator=mlp,
    param_distributions=param_dist,
    factor=3,
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Entrenamos con log-transform del target para mayor estabilidad
halving_search.fit(X_train, np.log1p(y_train))

best_model = halving_search.best_estimator_
best_params = halving_search.best_params_

# Predicciones en escala original
y_pred_train = np.expm1(best_model.predict(X_train))
y_pred_val = np.expm1(best_model.predict(X_val))

# Crear reporte
calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)
param_list["HalvingRandomSearchCV_MLP"] = best_params

print("Best parameters found:")
print(best_params)

n_iterations: 5
n_required_iterations: 5
n_possible_iterations: 7
min_resources_: 6
max_resources_: 11682
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 96
n_resources: 6
Fitting 3 folds for each of 96 candidates, totalling 288 fits


c:\Python312\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 96 is smaller than n_iter=1947. Running 96 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


----------
iter: 1
n_candidates: 32
n_resources: 18
Fitting 3 folds for each of 32 candidates, totalling 96 fits
----------
iter: 2
n_candidates: 11
n_resources: 54
Fitting 3 folds for each of 11 candidates, totalling 33 fits
----------
iter: 3
n_candidates: 4
n_resources: 162
Fitting 3 folds for each of 4 candidates, totalling 12 fits
----------
iter: 4
n_candidates: 2
n_resources: 486
Fitting 3 folds for each of 2 candidates, totalling 6 fits
Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   7.35e+13   |   1.25e+14
R²           |   0.8521     |   0.7438
MAE          |   4.36e+06   |   5.06e+06
EVS          |   0.8524     |   0.7441
Max Error    |   3.49e+08   |   2.41e+08
Median AE    |   2.66e+06   |   3.04e+06

Best parameters found:
{'solver': 'adam', 'max_iter': 1000, 'learning_rate_init': 0.001, 'learning_rate': 'adaptive', 'hidden_layer_sizes': (100, 50), 'alpha': 0.001, 'activation': 'relu'}


In [ ]:
input_example = X_train.iloc[0:1]

for name, params in param_list.items():
    if 'early_stopping' in params:
        params['early_stopping'] = True
    if 'validation_fraction' in params:
        params['validation_fraction'] = 0.1
    if 'n_iter_no_change' in params:
        params['n_iter_no_change'] = 10
    if 'random_state' in params:
        params['random_state'] = 42
    
    model = MLPRegressor(**params)

    # Entrenamos con log-transform del target
    model.fit(X_train, np.log1p(y_train))

    # Predicciones y destransformación
    y_pred_train = np.expm1(model.predict(X_train))
    y_pred_val = np.expm1(model.predict(X_val))

    metrics = calculate_metrics(y_val, y_pred_val, verbose=False)

    run_name = f"MLP_{name}"

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        signature = infer_signature(input_example, model.predict(input_example))
        mlflow.sklearn.log_model(model, artifact_path="model", input_example=input_example, signature=signature)


In [ ]:
# Hiperparámetros óptimos
params = {
    "solver": "adam",
    "max_iter": 1,  # solo un epoch por partial_fit
    "learning_rate_init": 0.005,
    "learning_rate": "adaptive",
    "hidden_layer_sizes": (1000, 50),
    "alpha": 0.01,
    "activation": "tanh",
    "warm_start": True,  # necesario para entrenar con partial_fit
    "random_state": 42
}

# Preparar target transformado
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

# Inicializar modelo
model = MLPRegressor(**params)

# Iniciar MLflow
with mlflow.start_run(run_name="MLP_regresor_por_epoch"):

    mlflow.log_params(params)

    # Entrenamiento por epoch manual
    n_epochs = 100
    for epoch in range(n_epochs):
        model.fit(X_train, y_train_log)

        # Calcular loss (RMSE) en train y val (en escala log)
        y_train_pred_log = model.predict(X_train)
        y_val_pred_log = model.predict(X_val)

        rmse_train = np.sqrt(mean_squared_error(y_train_log, y_train_pred_log))
        rmse_val = np.sqrt(mean_squared_error(y_val_log, y_val_pred_log))

        mlflow.log_metric("train_log_rmse", rmse_train, step=epoch)
        mlflow.log_metric("val_log_rmse", rmse_val, step=epoch)

    # Predicciones finales en escala original
    y_train_pred = np.expm1(model.predict(X_train))
    y_val_pred = np.expm1(model.predict(X_val))

    # Métricas finales
    metrics = {
        "mse_train": mean_squared_error(y_train, y_train_pred),
        "mse_val": mean_squared_error(y_val, y_val_pred),
        "r2_train": r2_score(y_train, y_train_pred),
        "r2_val": r2_score(y_val, y_val_pred),
        "mae_train": mean_absolute_error(y_train, y_train_pred),
        "mae_val": mean_absolute_error(y_val, y_val_pred),
        "median_ae_val": median_absolute_error(y_val, y_val_pred),
        "evs_val": explained_variance_score(y_val, y_val_pred)
    }

    mlflow.log_metrics(metrics)

    # Guardar modelo
    mlflow.sklearn.log_model(model, "mlp_model")

print("Entrenamiento y log completo.")


c:\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: M

🏃 View run MLP_regresor_por_epoch at: http://localhost:5000/#/experiments/841519844441403402/runs/7bbc503fc202458a8f74682cb7b47cd1
🧪 View experiment at: http://localhost:5000/#/experiments/841519844441403402
Entrenamiento y log completo.
